# Taller — Pokédex de Consola
**Módulo 1 · Introducción a APIs con Python**

---

En este taller vas a construir una **Pokédex funcional** que consulta datos reales desde internet.

Al terminar vas a tener un mini sistema con funciones reutilizables que:
- Busca cualquier Pokémon por nombre
- Muestra su ficha completa (tipo, stats, habilidades)
- Compara dos Pokémon por una stat específica
- Encuentra el Pokémon más fuerte de una lista

Este es exactamente el tipo de código que se usa en apps reales para consumir APIs de productos, usuarios, o servicios externos.


La API que usarás es **pública y gratuita** — no necesita registro ni contraseña: `https://pokeapi.co`

---
## Parte 1 — Conexión y exploración de la respuesta


Antes de construir funciones, necesitas entender qué devuelve la API.
El endpoint base es:
```
https://pokeapi.co/api/v2/pokemon/{nombre}
```
Reemplaza `{nombre}` con cualquier nombre en minúsculas: `pikachu`, `charizard`, `bulbasaur`.

In [7]:
import requests
import json

# Hacemos una primera consulta para explorar la respuesta
url = "https://pokeapi.co/api/v2/pokemon/pikachu"
response = requests.get(url)

# Verificamos que llegó bien
print("Status code:", response.status_code)

# Convertimos a diccionario
data = response.json()

# Veamos qué claves tiene este diccionario
print("\nClaves disponibles en la respuesta:")
print(list(data.keys()))



Status code: 200

Claves disponibles en la respuesta:
['abilities', 'base_experience', 'cries', 'forms', 'game_indices', 'height', 'held_items', 'id', 'is_default', 'location_area_encounters', 'moves', 'name', 'order', 'past_abilities', 'past_stats', 'past_types', 'species', 'sprites', 'stats', 'types', 'weight']


La respuesta tiene muchas claves. Las que vamos a usar en este taller son:

| Clave | Tipo | Qué contiene |
|---|---|---|
| `name` | string | Nombre del Pokémon |
| `id` | int | Número en la Pokédex |
| `height` | int | Altura (en decímetros × 10 = cm) |
| `weight` | int | Peso (en hectogramos × 10 = gramos) |
| `types` | lista de dicts | Tipos del Pokémon |
| `stats` | lista de dicts | Stats base (hp, attack, defense…) |
| `abilities` | lista de dicts | Habilidades |

In [8]:
# Exploremos cada sección antes de construir las funciones

# Datos básicos
print("Nombre:", data["name"])
print("ID:", data["id"])
print("Altura:", data["height"], "dm")
print("Peso:", data["weight"], "hg")

print()

# Tipos — data["types"] es una lista de diccionarios
# Cada elemento tiene la forma: {'slot': 1, 'type': {'name': 'electric', 'url': '...'}}
print("Estructura de types[0]:", data["types"][0])
print("Para obtener solo el nombre del tipo:", data["types"][0]["type"]["name"])

print()

# Stats — lista de diccionarios con 'stat' (nombre) y 'base_stat' (valor)
print("Estructura de stats[0]:", data["stats"][0])

print()

# Habilidades
print("Estructura de abilities[0]:", data["abilities"][0])

Nombre: pikachu
ID: 25
Altura: 4 dm
Peso: 60 hg

Estructura de types[0]: {'slot': 1, 'type': {'name': 'electric', 'url': 'https://pokeapi.co/api/v2/type/13/'}}
Para obtener solo el nombre del tipo: electric

Estructura de stats[0]: {'base_stat': 35, 'effort': 0, 'stat': {'name': 'hp', 'url': 'https://pokeapi.co/api/v2/stat/1/'}}

Estructura de abilities[0]: {'ability': {'name': 'static', 'url': 'https://pokeapi.co/api/v2/ability/9/'}, 'is_hidden': False, 'slot': 1}


### Ejercicio 1

Usando `data` (el diccionario que ya tienes), imprime en una sola celda:
1. Todos los tipos de Pikachu (puede tener más de uno)
2. Todas sus stats con su nombre y valor
3. Todas sus habilidades

Usa `for` para recorrer las listas. No hagas `data["types"][0]` a mano — recórrellas.

In [30]:
# Tu código aquí
# Pista: para types → for t in data["types"]: ... t["type"]["name"]

print("Todos los tipos de Pikachu (puede tener más de uno)")
for i in data["types"]:
    print("Tipo:", i["type"]["name"])

print("-"*30)
print("Todas sus stats con su nombre y valor")
print("")
for i in data["stats"]:
    print("Nombre:", i["stat"]["name"])
    print("Valor: ", i["base_stat"])
    
print("-"*30)
print("Todas sus habilidades")
print("")
for i in data["abilities"]:
    print("Habilidades: ", i["ability"]["name"])

    






Todos los tipos de Pikachu (puede tener más de uno)
Tipo: electric
------------------------------
Todas sus stats con su nombre y valor

Nombre: hp
Valor:  35
Nombre: attack
Valor:  55
Nombre: defense
Valor:  40
Nombre: special-attack
Valor:  50
Nombre: special-defense
Valor:  50
Nombre: speed
Valor:  90
------------------------------
Todas sus habilidades

Habilidades:  static
Habilidades:  lightning-rod


---
## Parte 2 — Función `buscar_pokemon()`

Ahora vamos a encapsular la consulta en una función reutilizable.

**¿Por qué una función?**
Porque en un sistema real no escribimos `requests.get(url)` cada vez que necesitamos un dato. Lo ponemos en una función, y desde cualquier parte del programa la llamamos con un nombre distinto. Si la URL de la API cambia algún día, solo cambiamos un lugar.

Esta función va a:
1. Recibir el nombre de un Pokémon como parámetro
2. Hacer la petición GET
3. Verificar el status_code
4. Retornar el diccionario de datos, o `None` si hubo error

In [38]:
import requests

def buscar_pokemon(nombre):
    """
    Consulta la PokéAPI y retorna los datos de un Pokémon.
    Retorna un diccionario si la consulta fue exitosa, None si no.
    """
    # Construimos la URL con el nombre que recibimos como parámetro
    # .lower() convierte el nombre a minúsculas — la API no acepta mayúsculas
    url = f"https://pokeapi.co/api/v2/pokemon/{nombre.lower()}"
    
    # Hacemos la petición
    response = requests.get(url)
    
    # Si el status_code es 200, la consulta fue exitosa
    if response.status_code == 200:
        return response.json()  # Retornamos el diccionario completo
    else:
        # Si no encontró el Pokémon (404) u otro error, avisamos y retornamos None
        print(f"No se encontró '{nombre}'. Código de error: {response.status_code}")
        return None


# Probamos la función
pikachu = buscar_pokemon("pikachu")
print("Nombre:", pikachu["name"])
print("ID:", pikachu["id"])

print()

# ¿Qué pasa si buscamos uno que no existe?
inexistente = buscar_pokemon("caterpie")
print("Resultado:", inexistente)  # Debería imprimir None

Nombre: pikachu
ID: 25

Resultado: {'abilities': [{'ability': {'name': 'shield-dust', 'url': 'https://pokeapi.co/api/v2/ability/19/'}, 'is_hidden': False, 'slot': 1}, {'ability': {'name': 'run-away', 'url': 'https://pokeapi.co/api/v2/ability/50/'}, 'is_hidden': True, 'slot': 3}], 'base_experience': 39, 'cries': {'latest': 'https://raw.githubusercontent.com/PokeAPI/cries/main/cries/pokemon/latest/10.ogg', 'legacy': 'https://raw.githubusercontent.com/PokeAPI/cries/main/cries/pokemon/legacy/10.ogg'}, 'forms': [{'name': 'caterpie', 'url': 'https://pokeapi.co/api/v2/pokemon-form/10/'}], 'game_indices': [{'game_index': 123, 'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}}, {'game_index': 123, 'version': {'name': 'blue', 'url': 'https://pokeapi.co/api/v2/version/2/'}}, {'game_index': 123, 'version': {'name': 'yellow', 'url': 'https://pokeapi.co/api/v2/version/3/'}}, {'game_index': 10, 'version': {'name': 'gold', 'url': 'https://pokeapi.co/api/v2/version/4/'}}, {'game

### Ejercicio 2

Usa `buscar_pokemon()` para buscar **tres Pokémon distintos** (los que quieras).

Para cada uno, imprime su nombre y su ID. Recuerda verificar que el resultado no sea `None` antes de acceder a los datos.

In [43]:
# Tu código aquí
# Pista: if resultado is not None: print(resultado["name"])

print("Escribe el nombre de 3 pokemones")
for i in range(3):
    poke = input("Escribe el nombre de un pokemon")
    pokemon1 = buscar_pokemon(poke)
    if pokemon1 is not None :
        print("Nombre: ", pokemon1["name"])
        print("ID: ", pokemon1["id"])
    else:
        print("No existe ese pokemon")

Escribe el nombre de 3 pokemones
Nombre:  charizard
ID:  6
Nombre:  pikachu
ID:  25
Nombre:  mew
ID:  151


---
## Parte 3 — Función `mostrar_ficha()`

Ahora que ya tenemos una función que trae los datos, necesitamos una que los **muestre de forma legible**.

Esta separación es intencional: `buscar_pokemon()` se encarga de *conseguir* los datos. `mostrar_ficha()` se encarga de *mostrarlos*. Si el día de mañana quisieras exportar la ficha a un archivo en vez de imprimirla, solo cambias esta función — la otra no la tocas.

La ficha debe mostrar:
- Nombre e ID
- Tipos
- Stats base
- Habilidades

In [44]:
def mostrar_ficha(data):
    """
    Recibe el diccionario de un Pokémon y muestra su ficha completa en consola.
    """
    # Verificamos que recibimos datos válidos (no None)
    if data is None:
        print("No hay datos para mostrar.")
        return  # Salimos de la función sin hacer nada más
    
    print("=" * 35)
    
    # Nombre en mayúsculas con el ID entre paréntesis
    print(f"  {data['name'].upper()} (#{data['id']})")
    
    print("=" * 35)
    
    # Tipos — recorremos la lista y extraemos solo el nombre de cada tipo
    tipos = [t["type"]["name"] for t in data["types"]]
    print(f"Tipo(s):      {' / '.join(tipos)}")  # join une los elementos con " / "
    
    # Altura y peso — la API los entrega en décimas, convertimos
    altura_cm = data["height"] * 10   # decímetros a centímetros
    peso_kg = data["weight"] / 10     # hectogramos a kilogramos
    print(f"Altura:       {altura_cm} cm")
    print(f"Peso:         {peso_kg} kg")
    
    print("-" * 35)
    print("  STATS BASE")
    print("-" * 35)
    
    # Stats — recorremos la lista e imprimimos nombre y valor de cada stat
    for stat in data["stats"]:
        nombre_stat = stat["stat"]["name"]
        valor = stat["base_stat"]
        # Barra visual proporcional: cada punto = un carácter "█"
        # Dividimos entre 2 para que no sea demasiado larga
        barra = "█" * (valor // 10)
        print(f"  {nombre_stat:<15} {valor:>3}  {barra}")
    
    print("-" * 35)
    print("  HABILIDADES")
    print("-" * 35)
    
    # Habilidades — recorremos y mostramos cada una
    for habilidad in data["abilities"]:
        nombre_hab = habilidad["ability"]["name"]
        # is_hidden indica si es una habilidad oculta
        oculta = " (oculta)" if habilidad["is_hidden"] else ""
        print(f"  - {nombre_hab}{oculta}")
    
    print("=" * 35)


# Probamos: buscamos y luego mostramos la ficha
charizard = buscar_pokemon("charizard")
mostrar_ficha(charizard)

  CHARIZARD (#6)
Tipo(s):      fire / flying
Altura:       170 cm
Peso:         90.5 kg
-----------------------------------
  STATS BASE
-----------------------------------
  hp               78  ███████
  attack           84  ████████
  defense          78  ███████
  special-attack  109  ██████████
  special-defense  85  ████████
  speed           100  ██████████
-----------------------------------
  HABILIDADES
-----------------------------------
  - blaze
  - solar-power (oculta)


### Ejercicio 3

Muestra la ficha completa de **dos Pokémon** que no hayamos usado todavía.

Luego responde en una celda de texto (Markdown):
> ¿Por qué es útil que `buscar_pokemon()` y `mostrar_ficha()` sean funciones separadas en vez de un solo bloque de código?

In [ ]:
# Tu código aquí

pokemon1 = input("Escribe el nombre de un Pokémon para mostrar su ficha: ")
pokemon_data = buscar_pokemon(pokemon1)
mostrar_ficha(pokemon_data)

pokemon2 = input("Escribe el nombre de otro Pokémon para mostrar su ficha: ")
pokemon_data = buscar_pokemon(pokemon2)
mostrar_ficha(pokemon_data)

# Es util tenerlas separada porque esto nos permite tener una mejor organización del código, y también nos permite reutilizar la función mostrar_ficha cada vez que queramos mostrar la ficha de un pokemon sin necesidad de repetir el código.

  POLIWAG (#60)
Tipo(s):      water
Altura:       60 cm
Peso:         12.4 kg
-----------------------------------
  STATS BASE
-----------------------------------
  hp               40  ████
  attack           50  █████
  defense          40  ████
  special-attack   40  ████
  special-defense  40  ████
  speed            90  █████████
-----------------------------------
  HABILIDADES
-----------------------------------
  - water-absorb
  - damp
  - swift-swim (oculta)
  POLIWRATH (#62)
Tipo(s):      water / fighting
Altura:       130 cm
Peso:         54.0 kg
-----------------------------------
  STATS BASE
-----------------------------------
  hp               90  █████████
  attack           95  █████████
  defense          95  █████████
  special-attack   70  ███████
  special-defense  90  █████████
  speed            70  ███████
-----------------------------------
  HABILIDADES
-----------------------------------
  - water-absorb
  - damp
  - swift-swim (oculta)


**Tu respuesta aquí:**

> _Escribe tu respuesta en esta celda._

---
## Parte 4 — Función `comparar_pokemon()`


Esta función recibe **dos nombres y una stat**, consulta ambos Pokémon, y dice cuál tiene el valor más alto en esa stat.

Las stats disponibles son:
`hp`, `attack`, `defense`, `special-attack`, `special-defense`, `speed`

Esta función tiene una dificultad extra: la stat viene como texto, pero el valor está dentro de una lista de diccionarios. Tendrás que recorrer `data["stats"]` para encontrar el que coincida con el nombre que pediste.

**Pista — cómo obtener el valor de una stat específica:**

```python
# Si tienes el diccionario de un Pokémon en `data`
# y quieres el valor de "speed":

for stat in data["stats"]:
    if stat["stat"]["name"] == "speed":
        valor = stat["base_stat"]
        break  # Ya encontramos lo que buscábamos, salimos del for
```

Tu función puede usar esta misma lógica internamente.

In [47]:
def obtener_stat(data, nombre_stat):
    """
    Función auxiliar: recibe el diccionario de un Pokémon y el nombre de una stat.
    Retorna el valor numérico de esa stat, o None si no existe.
    """
    for stat in data["stats"]:
        if stat["stat"]["name"] == nombre_stat:
            return stat["base_stat"]
    return None  # Si no encontró la stat, retorna None


def comparar_pokemon(nombre1, nombre2, stat):
    """
    Compara dos Pokémon en una stat específica e imprime cuál gana.
    Parámetros:
        nombre1, nombre2: nombres de los Pokémon a comparar
        stat: nombre de la stat a comparar (ej: "attack", "speed", "hp")
    """
    # Buscamos los dos Pokémon usando la función que ya construimos
    poke1 = buscar_pokemon(nombre1)
    poke2 = buscar_pokemon(nombre2)
    
    # Si alguno no se encontró, no podemos comparar
    if poke1 is None or poke2 is None:
        print("No se puede comparar: uno o ambos Pokémon no fueron encontrados.")
        return
    
    # Obtenemos el valor de la stat para cada uno
    valor1 = obtener_stat(poke1, stat)
    valor2 = obtener_stat(poke2, stat)
    
    # Si la stat no existe en ninguno de los dos
    if valor1 is None or valor2 is None:
        print(f"La stat '{stat}' no existe. Stats válidas: hp, attack, defense, special-attack, special-defense, speed")
        return
    
    # Mostramos los valores
    print(f"\n⚔️  Comparación de {stat.upper()}")
    print(f"  {poke1['name'].capitalize():<15} {valor1}")
    print(f"  {poke2['name'].capitalize():<15} {valor2}")
    print()
    
    # Determinamos el ganador
    if valor1 > valor2:
        print(f"🏆 Gana: {poke1['name'].capitalize()} ({valor1} vs {valor2})")
    elif valor2 > valor1:
        print(f"🏆 Gana: {poke2['name'].capitalize()} ({valor2} vs {valor1})")
    else:
        print(f"🤝 Empate — ambos tienen {valor1} en {stat}")


# Probamos
comparar_pokemon("pikachu", "bulbasaur", "speed")
comparar_pokemon("mewtwo", "gengar", "attack")


⚔️  Comparación de SPEED
  Pikachu         90
  Bulbasaur       45

🏆 Gana: Pikachu (90 vs 45)

⚔️  Comparación de ATTACK
  Mewtwo          110
  Gengar          65

🏆 Gana: Mewtwo (110 vs 65)


### Ejercicio 4

Completa los siguientes tres comparativos. Elige tú la stat para cada uno:

1. Compara `snorlax` vs `machamp` en la stat que consideres más relevante para ellos. Justifica tu elección en un comentario.
2. Compara dos Pokémon de tu elección en `defense`.
3. Intenta comparar con una stat que **no exista** (por ejemplo `"fuerza"`). ¿Qué imprime la función? ¿Por qué?

In [ ]:
# Tu código aquí

comparar_pokemon("snorlax", "machamp", "hp")   #Me da curiosidad saber quien tiene mas vida porque snorlax es grande y creo que tiene mas vida
print("=="*10)

comparar_pokemon("charizard", "dragonite", "defense")
print("=="*10)

comparar_pokemon("gengar", "alakazam", "fuerza") # Sale que no funciona porque no esta dentro del diccionario por lo tanto no va a dar respuesta y como esta protegido de error no va a romper el programa


⚔️  Comparación de HP
  Snorlax         160
  Machamp         90

🏆 Gana: Snorlax (160 vs 90)

⚔️  Comparación de DEFENSE
  Charizard       78
  Dragonite       95

🏆 Gana: Dragonite (95 vs 78)
La stat 'fuerza' no existe. Stats válidas: hp, attack, defense, special-attack, special-defense, speed


---
## Parte 5 — Función `pokemon_mas_fuerte()`

Esta es la función más compleja del taller. Recibe una **lista de nombres** y una **stat**, consulta todos, y retorna el nombre del Pokémon con el valor más alto en esa stat.

Esta función integra todo lo que construiste:
- Usa `buscar_pokemon()` para cada nombre de la lista
- Usa `obtener_stat()` para extraer el valor
- Recorre la lista para encontrar el máximo
- Retorna el ganador

In [49]:
def pokemon_mas_fuerte(lista_nombres, stat):
    """
    Recibe una lista de nombres de Pokémon y una stat.
    Retorna el nombre del Pokémon con el valor más alto en esa stat.
    """
    # Estas variables van a guardar el mejor resultado encontrado hasta ahora
    mejor_nombre = None
    mejor_valor = -1  # Empezamos en -1 para que cualquier valor real lo supere
    
    print(f"Buscando el más fuerte en '{stat}'...\n")
    
    # Recorremos cada nombre de la lista 
    for nombre in lista_nombres:
        
        # Consultamos la API para este Pokémon
        data = buscar_pokemon(nombre)
        
        # Si no se encontró, lo saltamos y seguimos con el siguiente
        if data is None:
            continue
        
        # Obtenemos el valor de la stat
        valor = obtener_stat(data, stat)
        
        # Si la stat no existe para este Pokémon, lo saltamos
        if valor is None:
            continue
        
        # Mostramos el resultado parcial para que el usuario vea el progreso
        print(f"  {nombre.capitalize():<15} {stat}: {valor}")
        
        # ¿Es el mejor hasta ahora?
        if valor > mejor_valor:
            mejor_valor = valor
            mejor_nombre = nombre
    
    # Al terminar el for, tenemos el ganador
    print()
    if mejor_nombre is not None:
        print(f"El más fuerte en {stat.upper()} es: {mejor_nombre.upper()} con {mejor_valor} puntos")
    else:
        print("No se pudo determinar un ganador.")
    
    return mejor_nombre


# Probamos con una lista
equipo = ["pikachu", "charizard", "mewtwo", "snorlax", "gengar"]
ganador = pokemon_mas_fuerte(equipo, "speed")

Buscando el más fuerte en 'speed'...

  Pikachu         speed: 90
  Charizard       speed: 100
  Mewtwo          speed: 130
  Snorlax         speed: 30
  Gengar          speed: 110

El más fuerte en SPEED es: MEWTWO con 130 puntos


### Ejercicio 5 — Desafío final

1. Arma tu propio equipo de **6 Pokémon** (los que quieras) en una lista.
2. Encuentra cuál es el más fuerte en **`attack`**.
3. Encuentra cuál es el más fuerte en **`defense`**.
4. Muestra la ficha completa del Pokémon ganador en `attack` usando `mostrar_ficha()`.

**Restricción:** Todo debe hacerse en una sola celda, usando las funciones que ya construiste. No hagas peticiones GET directamente — usa `buscar_pokemon()`.

In [51]:
# Tu código aquí

# 1. Define tu equipo
mi_equipo = ["pikachu", "gengar", "dragonite", "lucario", "Sirfetch'd", "dracovish"]

# 2. Encuentra el más fuerte en attack
ganador_attack = pokemon_mas_fuerte(mi_equipo, "attack")

# 3. Encuentra el más fuerte en defense
ganador_defense = pokemon_mas_fuerte(mi_equipo, "defense")

# 4. Muestra la ficha del ganador en attack
ganador_attack_ficha = mostrar_ficha(buscar_pokemon(ganador_attack))


Buscando el más fuerte en 'attack'...

  Pikachu         attack: 55
  Gengar          attack: 65
  Dragonite       attack: 134
  Lucario         attack: 110
No se encontró 'Sirfetch'd'. Código de error: 400
  Dracovish       attack: 90

El más fuerte en ATTACK es: DRAGONITE con 134 puntos
Buscando el más fuerte en 'defense'...

  Pikachu         defense: 40
  Gengar          defense: 60
  Dragonite       defense: 95
  Lucario         defense: 70
No se encontró 'Sirfetch'd'. Código de error: 400
  Dracovish       defense: 100

El más fuerte en DEFENSE es: DRACOVISH con 100 puntos
  DRAGONITE (#149)
Tipo(s):      dragon / flying
Altura:       220 cm
Peso:         210.0 kg
-----------------------------------
  STATS BASE
-----------------------------------
  hp               91  █████████
  attack          134  █████████████
  defense          95  █████████
  special-attack  100  ██████████
  special-defense 100  ██████████
  speed            80  ████████
---------------------------------

---
## ¿Dónde aparece esto en la vida real?

Lo que construiste hoy no es solo un juego. Es exactamente el patrón que usan los sistemas reales:

| Lo que hiciste aquí | Equivalente en producción |
|---|---|
| `buscar_pokemon(nombre)` | `buscar_producto(id)`, `obtener_usuario(email)` |
| `mostrar_ficha(data)` | `renderizar_tarjeta(producto)`, `imprimir_perfil(usuario)` |
| `comparar_pokemon()` | Comparador de precios, comparador de candidatos en un ATS |
| `pokemon_mas_fuerte()` | Algoritmo de ranking, recomendación del mejor plan, filtro del producto más vendido |
| Verificar `status_code` antes de procesar | Manejo de errores en cualquier integración con API externa |

La estructura de **funciones con responsabilidad única** que usamos hoy es uno de los principios más importantes del desarrollo real. Cada función hace una sola cosa — y eso las hace reutilizables, fáciles de probar y fáciles de modificar.

---

**API usada:** PokéAPI — `https://pokeapi.co` (gratuita, sin autenticación, documentación pública)

---

*© 2026 Ana Alvarado · Educadora Techy & Desarrolladora Full Stack · Todos los derechos reservados. linkedin.com/in/ana-alvarado-instructora-full-stack*